# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a dataset using the `mlcroissant` library. The dataset schema follows the Croissant standard and is accessible via a URL.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Install mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")
print(f"License: {metadata.license}")

# Print keywords
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else ''}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print("Record sets in dataset:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '')}")

# For each record set, show fields and columns referenced by their @id
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}, name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # single field case
        fields = [fields]
    print("  Fields:")
    for field in fields:
        fid = field.get('@id') if isinstance(field, dict) else field
        print(f"    - {fid}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("  Columns:")
        for col in columns:
            cid = col.get('@id') if isinstance(col, dict) else col
            print(f"    - {cid}")

In [ ]:
# Example: Print first 3 records from each record set, referencing record_set @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        for rec in records[:3]:
            print(rec)
    except Exception as e:
        print(f"  Unable to load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to load dataframes from each record set
dataframes = {}

# Get all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {rs_id}, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Error loading DataFrame for RecordSet @id: {rs_id}: {e}")

# Pick the first record set (or one with data) for further demonstration
if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"\nShowing first 5 rows for RecordSet @id: {example_rs_id}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Demonstrate using field/column `@id`s.

In [ ]:
# Select a record set and a numeric column for filtering
rs_id = example_rs_id
df = dataframes[rs_id]

# Find numeric columns automatically, referencing by @id
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {rs_id}: {numeric_fields}")

# If no numeric field found, skip the rest
if numeric_fields:
    numeric_field_id = numeric_fields[0]

    # Threshold-based filtering
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical field
    cat_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
    group_field_id = cat_fields[0] if cat_fields else None

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Here we use matplotlib/seaborn to plot the distribution of the numeric field and its grouping by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet @id {rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by categorical group
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}' for RecordSet @id {rs_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library for loading, inspecting, and processing a dataset defined with the Croissant schema. Using entity `@id` references, we explored record sets, fields, and columns, loaded dataframes, performed simple EDA steps (filtering, normalization, grouping), and visualized patterns in the data.

For further analysis, consult the dataset documentation and Croissant schema for field details and explore additional statistical and visualization techniques.